## 1、加载数据集

In [1]:

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

In [2]:
datasets = load_dataset("qgyd2021/sentence_pair", name='lcqmc',cache_dir="../data",trust_remote_code=True, split = 'train')
datasets

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'category', 'data_source', 'split'],
    num_rows: 238766
})

In [3]:
for i in range(10):
    print("sentence1: {s1}, sentence2: {s2}, label: {l}".format(s1=datasets[i]['sentence1'], s2=datasets[i]['sentence2'], l=datasets[i]['label']))

sentence1: 喜欢打篮球的男生喜欢什么样的女生, sentence2: 爱打篮球的男生喜欢什么样的女生, label: 1
sentence1: 我手机丢了，我想换个手机, sentence2: 我想买个新手机，求推荐, label: 1
sentence1: 大家觉得她好看吗, sentence2: 大家觉得跑男好看吗？, label: 0
sentence1: 求秋色之空漫画全集, sentence2: 求秋色之空全集漫画, label: 1
sentence1: 晚上睡觉带着耳机听音乐有什么害处吗？, sentence2: 孕妇可以戴耳机听音乐吗?, label: 0
sentence1: 学日语软件手机上的, sentence2: 手机学日语的软件, label: 1
sentence1: 打印机和电脑怎样连接，该如何设置, sentence2: 如何把带无线的电脑连接到打印机上, label: 0
sentence1: 侠盗飞车罪恶都市怎样改车, sentence2: 侠盗飞车罪恶都市怎么改车, label: 1
sentence1: 什么花一年四季都开, sentence2: 什么花一年四季都是开的, label: 1
sentence1: 看图猜一电影名, sentence2: 看图猜电影！, label: 1


## 2、数据预处理

In [4]:
datasets = datasets.train_test_split(test_size=0.2)
datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'category', 'data_source', 'split'],
        num_rows: 191012
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'category', 'data_source', 'split'],
        num_rows: 47754
    })
})

In [5]:
datasets['train'][0]

{'sentence1': '怎么才可以去除腋毛',
 'sentence2': '腋毛可以去除吗',
 'label': '0',
 'category': None,
 'data_source': 'lcqmc',
 'split': 'train'}

In [6]:
import torch

tokenizer = AutoTokenizer.from_pretrained('bert-base-chinese', cache_dir='./models')

def process_function(datasets):
    tokenized_datasets = tokenizer(datasets["sentence1"], datasets["sentence2"], max_length=128, truncation=True)
    tokenized_datasets["labels"] = [float(label) for label in datasets["label"]]
    return tokenized_datasets


d:\program\anaconda3\envs\transformers\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [7]:

tokenized_datasets = datasets.map(process_function, batched=True, remove_columns=datasets["train"].column_names)
tokenized_datasets

Map:   0%|          | 0/191012 [00:00<?, ? examples/s]

Map:   0%|          | 0/47754 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 191012
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 47754
    })
})

## 3、创建模型

In [8]:
from transformers import AutoModelForSequenceClassification 
model = AutoModelForSequenceClassification.from_pretrained("bert-base-chinese", cache_dir='./models', num_labels=1)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 4、评估函数

In [9]:
import evaluate

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

In [10]:
def eval_metric(eval_predict):
    predictions, labels = eval_predict
    predictions = [int(p > 0.5) for p in predictions]
    labels = [int(l) for l in labels]
    # predictions = predictions.argmax(axis=-1)
    acc = acc_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    acc.update(f1)
    return acc


## 5、创建训练器

In [11]:
train_args = TrainingArguments(
    output_dir="./sentence_similarity_model",      # 输出文件夹
    per_device_train_batch_size=64,  # 训练时的batch_size
    per_device_eval_batch_size=128,   # 验证时的batch_size
    logging_steps=10,                # log 打印的频率
    eval_strategy="epoch",           # 评估策略
    save_strategy="epoch",           # 保存策略
    save_total_limit=3,              # 最大保存数
    metric_for_best_model="f1",      # 设定评估指标
    num_train_epochs=3,              # 训练轮数, 默认为3
    report_to=['tensorboard'],       # tensorboard展示结果
    load_best_model_at_end=True)     # 训练完成后加载最优模型
train_args

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
evaluation_s

## 6、开始训练

In [12]:
from transformers import DataCollatorWithPadding
trainer = Trainer(model=model, 
                  args=train_args, 
                  tokenizer=tokenizer,
                  train_dataset=tokenized_datasets["train"], 
                  eval_dataset=tokenized_datasets["test"], 
                  data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
                  compute_metrics=eval_metric)

In [13]:
trainer.train()

  0%|          | 0/8955 [00:00<?, ?it/s]

{'loss': 0.3921, 'grad_norm': 5.479007244110107, 'learning_rate': 4.994416527079844e-05, 'epoch': 0.0}
{'loss': 0.1957, 'grad_norm': 3.9593281745910645, 'learning_rate': 4.988833054159688e-05, 'epoch': 0.01}
{'loss': 0.2212, 'grad_norm': 3.48579478263855, 'learning_rate': 4.983249581239531e-05, 'epoch': 0.01}
{'loss': 0.1355, 'grad_norm': 6.748331546783447, 'learning_rate': 4.977666108319375e-05, 'epoch': 0.01}
{'loss': 0.1868, 'grad_norm': 22.160093307495117, 'learning_rate': 4.972082635399218e-05, 'epoch': 0.02}
{'loss': 0.1814, 'grad_norm': 4.959961891174316, 'learning_rate': 4.9664991624790626e-05, 'epoch': 0.02}
{'loss': 0.131, 'grad_norm': 10.963081359863281, 'learning_rate': 4.960915689558906e-05, 'epoch': 0.02}
{'loss': 0.1363, 'grad_norm': 7.2528300285339355, 'learning_rate': 4.955332216638749e-05, 'epoch': 0.03}
{'loss': 0.1152, 'grad_norm': 6.872032165527344, 'learning_rate': 4.949748743718593e-05, 'epoch': 0.03}
{'loss': 0.1262, 'grad_norm': 6.25338888168335, 'learning_rate

  0%|          | 0/374 [00:00<?, ?it/s]

C:\Users\Administrator\AppData\Local\Temp\ipykernel_7360\1570083857.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predictions = [int(p > 0.5) for p in predictions]


{'eval_loss': 0.06627977639436722, 'eval_accuracy': 0.9121958369979478, 'eval_f1': 0.9245388283991721, 'eval_runtime': 642.9064, 'eval_samples_per_second': 74.278, 'eval_steps_per_second': 0.582, 'epoch': 1.0}
{'loss': 0.0596, 'grad_norm': 1.194163203239441, 'learning_rate': 3.330541596873255e-05, 'epoch': 1.0}
{'loss': 0.0645, 'grad_norm': 1.4878588914871216, 'learning_rate': 3.324958123953099e-05, 'epoch': 1.01}
{'loss': 0.0653, 'grad_norm': 2.4458131790161133, 'learning_rate': 3.319374651032943e-05, 'epoch': 1.01}
{'loss': 0.0578, 'grad_norm': 1.0024064779281616, 'learning_rate': 3.3137911781127865e-05, 'epoch': 1.01}
{'loss': 0.0577, 'grad_norm': 1.0800905227661133, 'learning_rate': 3.30820770519263e-05, 'epoch': 1.02}
{'loss': 0.0616, 'grad_norm': 0.8983327150344849, 'learning_rate': 3.302624232272474e-05, 'epoch': 1.02}
{'loss': 0.0508, 'grad_norm': 2.2804532051086426, 'learning_rate': 3.2970407593523174e-05, 'epoch': 1.02}
{'loss': 0.054, 'grad_norm': 2.941087484359741, 'learnin

...

{'eval_loss': 0.061743926256895065, 'eval_accuracy': 0.922498638857478, 'eval_f1': 0.9339896909055238, 'eval_runtime': 941.8309, 'eval_samples_per_second': 50.703, 'eval_steps_per_second': 0.397, 'epoch': 2.0}
{'loss': 0.036, 'grad_norm': 0.832237720489502, 'learning_rate': 1.6610831937465104e-05, 'epoch': 2.0}
{'loss': 0.0383, 'grad_norm': 1.004292607307434, 'learning_rate': 1.655499720826354e-05, 'epoch': 2.01}
{'loss': 0.0371, 'grad_norm': 1.2501994371414185, 'learning_rate': 1.6499162479061976e-05, 'epoch': 2.01}
{'loss': 0.0314, 'grad_norm': 0.8924177289009094, 'learning_rate': 1.6443327749860412e-05, 'epoch': 2.01}
{'loss': 0.0288, 'grad_norm': 0.9936417937278748, 'learning_rate': 1.6387493020658852e-05, 'epoch': 2.02}
{'loss': 0.0376, 'grad_norm': 1.0412962436676025, 'learning_rate': 1.6331658291457288e-05, 'epoch': 2.02}
{'loss': 0.038, 'grad_norm': 0.6346471309661865, 'learning_rate': 1.6275823562255724e-05, 'epoch': 2.02}
{'loss': 0.0459, 'grad_norm': 1.2280795574188232, 'lea

  0%|          | 0/374 [00:00<?, ?it/s]

C:\Users\Administrator\AppData\Local\Temp\ipykernel_7360\1570083857.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predictions = [int(p > 0.5) for p in predictions]


{'eval_loss': 0.06305546313524246, 'eval_accuracy': 0.9234200276416635, 'eval_f1': 0.9343317351721165, 'eval_runtime': 945.7289, 'eval_samples_per_second': 50.494, 'eval_steps_per_second': 0.395, 'epoch': 3.0}
{'train_runtime': 6841.4721, 'train_samples_per_second': 83.759, 'train_steps_per_second': 1.309, 'train_loss': 0.06089593756970981, 'epoch': 3.0}


TrainOutput(global_step=8955, training_loss=0.06089593756970981, metrics={'train_runtime': 6841.4721, 'train_samples_per_second': 83.759, 'train_steps_per_second': 1.309, 'total_flos': 1.7499038507178336e+16, 'train_loss': 0.06089593756970981, 'epoch': 3.0})

## 7、评估

In [14]:
trainer.evaluate(tokenized_datasets["test"])

  0%|          | 0/374 [00:00<?, ?it/s]

C:\Users\Administrator\AppData\Local\Temp\ipykernel_7360\1570083857.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predictions = [int(p > 0.5) for p in predictions]


{'eval_loss': 0.06305546313524246,
 'eval_accuracy': 0.9234200276416635,
 'eval_f1': 0.9343317351721165,
 'eval_runtime': 962.7601,
 'eval_samples_per_second': 49.601,
 'eval_steps_per_second': 0.388,
 'epoch': 3.0}

## 8、预测

In [15]:
from transformers import pipeline, TextClassificationPipeline
model.config.id2label = {0: "不相似", 1: "相似"}

In [16]:
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

In [18]:
result1 = pipe({"text": "杭州是个好地方", "text_pair": "杭州真好"})
result2 = pipe({"text": "杭州是个好地方", "text_pair": "我喜欢今天的天气"})

In [19]:
result1

{'label': '不相似', 'score': 1.0}

In [22]:
result1 = pipe({"text": "杭州是个好地方", "text_pair": "杭州这个地方真好"}, function_to_apply="none")
result2 = pipe({"text": "杭州是个好地方", "text_pair": "我喜欢今天的天气"}, function_to_apply="none")
(result1,result2)

({'label': '不相似', 'score': 0.7523335814476013},
 {'label': '不相似', 'score': -0.004789017606526613})

In [25]:
result1["label"] = "相似" if result1["score"] > 0.5 else "不相似"
result1

{'label': '相似', 'score': 0.7523335814476013}

In [27]:
result2["label"] = "相似" if result2["score"] > 0.5 else "不相似"
result2

{'label': '不相似', 'score': -0.004789017606526613}